In [ ]:
!pip install -q pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 70.4 MB/s eta 0:00:00


In [ ]:
import pdfplumber
import pandas as pd
import numpy as np
from tqdm import tqdm

# Helper Function

In [ ]:
def group_by_vertical_bbox(words, y_tol=1):
    rows = []

    sorted_words = sorted(words, key=lambda w: w["top"])

    for word in sorted_words:
        matched = False
        for row in rows:
            if (abs(word["top"] - row["top"]) <= y_tol and abs(word["bottom"] - row["bottom"]) <= y_tol):
                row["words"].append(word)
                matched = True
                break

        if not matched:
            rows.append({
                "top": word["top"],
                "bottom": word["bottom"],
                "words": [word],
            })

    return rows

In [ ]:
def collect_vertical_gaps(rows):
    gaps = []

    rows = sorted(rows, key=lambda r: r["top"])
    gaps.extend(rows[i + 1]["top"] - rows[i]["bottom"] for i in range(len(rows) - 1))

    return gaps

In [ ]:
def estimate_gap_threshold(gaps, factor=2.5):
    gaps = np.array([g for g in gaps if g > 0])

    if len(gaps) == 0:
        return None

    binned = np.round(gaps, 1)
    counts = np.bincount((binned * 10).astype(int))

    if len(counts) == 0:
        return None

    mode_gap = counts.argmax() / 10
    return factor * mode_gap

In [ ]:
def zoning_from_words(words, gap_threshold):
    if not words:
        return []

    words = sorted(words, key=lambda w: w["top"])

    zones = []
    start = words[0]["top"]

    for i in range(1, len(words)):
        gap = words[i]["top"] - words[i-1]["bottom"]
        if gap_threshold and gap > gap_threshold:
            zones.append((start, words[i-1]["bottom"]))
            start = words[i]["top"]

    zones.append((start, words[-1]["bottom"]))
    return zones

In [ ]:
def extract_zone_text(words, y0, y1):
    zone_words = [
        w for w in words
        if w["top"] >= y0 and w["bottom"] <= y1
    ]
    return " ".join(w["text"] for w in zone_words)

# Main Loop

In [ ]:
FILE_RULES = {
    "PT AKR Corporindo Tbk": {
        "start" : 158,
        "end"   : 268,
    },
    "PT Bank Central Asia Tbk": {
        "start" : 284,
        "end"   : 531,
    },
    "PT Bank CIMB Niaga Tbk": {
        "start" : 406,
        "end"   : 614,
    },
    "PT Bank Jago Tbk": {
        "start" : 116,
        "end"   : 266,
    },
    "PT Bank Mandiri Tbk": {
        "start" : 562,
        "end"   : 1152,
    },
    "PT Bank Negara Indonesia Tbk": {
        "start" : 606,
        "end"   : 1154,
    },
    "PT Bank Rakyat Indonesia Tbk": {
        "start" : 459,
        "end"   : 813,
    },
    "PT Indosat Tbk": {
        "start" : 200,
        "end"   : 285,
    },
    "PT Telkom Indonesia Tbk": {
        "start" : 180,
        "end"   : 366,
    },
    "PT Wijaya Karya Tbk": {
        "start" : 422,
        "end"   : 826,
    },
}

In [ ]:
for key, value in tqdm(FILE_RULES.items()):
    result = []

    with pdfplumber.open(f"{key}.pdf") as pdf:
        for page_num, page in enumerate(pdf.pages[value["start"] - 1:value["end"]], start=value["start"]):
            words = page.extract_words(use_text_flow=True)
            mid_x = page.width / 2

            left_words  = [w for w in words if w["x0"] < mid_x]
            right_words = [w for w in words if w["x0"] >= mid_x]

            for side, col_words in [("left", left_words), ("right", right_words)]:

                rows = group_by_vertical_bbox(col_words)
                gaps = collect_vertical_gaps(rows)
                gap_threshold = estimate_gap_threshold(gaps)

                zones = zoning_from_words(col_words, gap_threshold)

                for zone_id, (y0, y1) in enumerate(zones):
                    text = extract_zone_text(col_words, y0, y1)
                    if not text.strip():
                        continue

                    result.append({
                        "file": key,
                        "page": page_num,
                        "side": side,
                        "zone": zone_id,
                        "text": text
                    })

    df = pd.DataFrame(result)
    df.to_excel(f"{key}.xlsx", index=False,)
    print(f" - {key} done")

 10%|█         | 1/10 [00:21<03:13, 21.45s/it]

 - PT AKR Corporindo Tbk done


 20%|██        | 2/10 [01:04<04:34, 34.30s/it]

 - PT Bank Central Asia Tbk done


 30%|███       | 3/10 [02:04<05:22, 46.14s/it]

...

 - PT Indosat Tbk done


 90%|█████████ | 9/10 [09:14<00:56, 56.16s/it]

 - PT Telkom Indonesia Tbk done


100%|██████████| 10/10 [11:22<00:00, 68.22s/it]

 - PT Wijaya Karya Tbk done
